# Auditoria e baseline da TDA

Notebook de auditoria reproduzível para a Entrega 4. Execute as células em ordem.
Os caminhos são relativos à raiz do repositório. A base Silver deve existir localmente.


In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

RAIZ = Path.cwd()
if not (RAIZ / "data").exists() and (RAIZ.parent / "data").exists():
    RAIZ = RAIZ.parent
ARQUIVO = RAIZ / "data/silver/fluxo_educacao_superior/indicadores_trajetoria_educacao_superior_2020_2024.parquet"
assert ARQUIVO.exists(), f"Arquivo não encontrado: {ARQUIVO}"
df = pd.read_parquet(ARQUIVO)
print("Dimensão:", df.shape)
display(df.head())


Dimensão: (174240, 31)


,CO_IES,NO_IES,TP_CATEGORIA_ADMINISTRATIVA,TP_ORGANIZACAO_ACADEMICA,CO_CURSO,NO_CURSO,CO_REGIAO,CO_UF,CO_MUNICIPIO,TP_GRAU_ACADEMICO,...,QT_INGRESSANTE,QT_PERMANENCIA,QT_CONCLUINTE,QT_DESISTENCIA,QT_FALECIDO,TAP,TCA,TDA,TCAN,TADA
0,1,UNIVERSIDADE FEDERAL DE MATO GROSSO,1.0,1.0,1.0,DIREITO,5.0,51.0,5103403.0,1.0,...,100.0,100.0,0.0,0.0,0.0,100.0,0.0,0.0,0.0,0.0
1,1,UNIVERSIDADE FEDERAL DE MATO GROSSO,1.0,1.0,1.0,DIREITO,5.0,51.0,5103403.0,1.0,...,100.0,97.0,0.0,3.0,0.0,97.0,0.0,3.0,0.0,3.0
2,1,UNIVERSIDADE FEDERAL DE MATO GROSSO,1.0,1.0,1.0,DIREITO,5.0,51.0,5103403.0,1.0,...,100.0,92.0,1.0,4.0,0.0,92.0,1.0,7.0,1.0,4.0
3,1,UNIVERSIDADE FEDERAL DE MATO GROSSO,1.0,1.0,1.0,DIREITO,5.0,51.0,5103403.0,1.0,...,100.0,91.0,1.0,0.0,0.0,91.0,2.0,7.0,1.0,0.0
4,1,UNIVERSIDADE FEDERAL DE MATO GROSSO,1.0,1.0,1.0,DIREITO,5.0,51.0,5103403.0,1.0,...,100.0,34.0,57.0,0.0,0.0,34.0,59.0,7.0,57.0,0.0


In [2]:
print("Tipos de dados:")
display(df.dtypes.astype(str).rename("tipo").to_frame())
print("Ausências por variável:")
display(df.isna().sum().sort_values(ascending=False).rename("ausencias").to_frame())
print("Registros por ano de referência:")
display(df["NU_ANO_REFERENCIA"].value_counts(dropna=False).sort_index().rename("registros").to_frame())


Tipos de dados:


,tipo
CO_IES,int64
NO_IES,object
TP_CATEGORIA_ADMINISTRATIVA,float64
TP_ORGANIZACAO_ACADEMICA,float64
CO_CURSO,float64
NO_CURSO,object
CO_REGIAO,float64
CO_UF,float64
CO_MUNICIPIO,float64
TP_GRAU_ACADEMICO,float64


Ausências por variável:


,ausencias
CO_REGIAO,26230
CO_MUNICIPIO,26230
CO_UF,26230
TP_CATEGORIA_ADMINISTRATIVA,0
TP_ORGANIZACAO_ACADEMICA,0
NO_IES,0
CO_IES,0
NO_CURSO,0
CO_CURSO,0
TP_GRAU_ACADEMICO,0


Registros por ano de referência:


,registros
NU_ANO_REFERENCIA,
2020.0,34848
2021.0,34848
2022.0,34848
2023.0,34848
2024.0,34848


In [3]:
chave = ["CO_IES", "CO_CURSO", "NU_ANO_INGRESSO", "NU_ANO_REFERENCIA"]
print("Duplicatas na chave candidata:", int(df.duplicated(chave).sum()))
print("TDA ausente:", int(df["TDA"].isna().sum()))
print("TDA abaixo de 0:", int((df["TDA"] < 0).sum()))
print("TDA acima de 100:", int((df["TDA"] > 100).sum()))
display(df["TDA"].describe().to_frame())


Duplicatas na chave candidata: 0
TDA ausente: 0
TDA abaixo de 0: 0
TDA acima de 100: 0


,TDA
count,174240.000000
mean,40.782347
std,28.359274
min,0.000000
25%,16.363636
50%,39.215686
75%,62.000000
max,100.000000


## Preparação e partição

A partição é temporal: anos anteriores a 2024 para treino e 2024 para teste.
Isso imita uma tentativa de prever registros de 2024 com dados de anos anteriores,
mas não impede a presença dos mesmos cursos ou coortes nos dois conjuntos.


In [4]:
categoricas = [
    "TP_CATEGORIA_ADMINISTRATIVA", "TP_ORGANIZACAO_ACADEMICA",
    "TP_GRAU_ACADEMICO", "TP_MODALIDADE_ENSINO",
    "CO_REGIAO", "CO_UF", "CO_CINE_AREA_GERAL"
]
numericas = ["NU_ANO_INGRESSO", "QT_INGRESSANTE"]
alvo = "TDA"
base = df[categoricas + numericas + ["NU_ANO_REFERENCIA", alvo]].copy()
base = base.dropna(subset=[alvo, "NU_ANO_REFERENCIA"])
base = base[base[alvo].between(0, 100)]
treino = base[base["NU_ANO_REFERENCIA"] < 2024].copy()
teste = base[base["NU_ANO_REFERENCIA"] == 2024].copy()
print("Base modelagem:", base.shape)
print("Treino:", treino.shape, "| Teste:", teste.shape)
assert len(treino) > 0 and len(teste) > 0


Base modelagem: (174240, 11)
Treino: (139392, 11) | Teste: (34848, 11)


In [5]:
preprocessamento = ColumnTransformer([
    ("categoricas", Pipeline([
        ("imputar", SimpleImputer(strategy="most_frequent")),
        ("codificar", OneHotEncoder(handle_unknown="ignore"))
    ]), categoricas),
    ("numericas", Pipeline([
        ("imputar", SimpleImputer(strategy="median"))
    ]), numericas)
])
resultados = []
for nome, estimador in {
    "baseline_media": DummyRegressor(strategy="mean"),
    "random_forest": RandomForestRegressor(
        n_estimators=100, min_samples_leaf=5, random_state=42, n_jobs=-1
    )
}.items():
    modelo = Pipeline([
        ("preprocessamento", preprocessamento),
        ("modelo", estimador)
    ])
    modelo.fit(treino[categoricas + numericas], treino[alvo])
    previsoes = modelo.predict(teste[categoricas + numericas])
    resultados.append({
        "modelo": nome,
        "treino_registros": len(treino),
        "teste_registros": len(teste),
        "MAE": float(mean_absolute_error(teste[alvo], previsoes)),
        "RMSE": float(np.sqrt(mean_squared_error(teste[alvo], previsoes))),
        "R2": float(r2_score(teste[alvo], previsoes))
    })
display(pd.DataFrame(resultados))


,modelo,treino_registros,teste_registros,MAE,RMSE,R2
0,baseline_media,139392,34848,27.667629,32.821726,-0.852685
1,random_forest,139392,34848,23.814402,27.750964,-0.324448


In [6]:
saida = RAIZ / "relatorios" / "auditoria_notebook.json"
saida.parent.mkdir(parents=True, exist_ok=True)
relatorio = {
    "arquivo": str(ARQUIVO.relative_to(RAIZ)),
    "dimensao_silver": list(df.shape),
    "duplicatas_chave_candidata": int(df.duplicated(
        ["CO_IES", "CO_CURSO", "NU_ANO_INGRESSO", "NU_ANO_REFERENCIA"]
    ).sum()),
    "ausencias_por_coluna": {k: int(v) for k, v in df.isna().sum().items()},
    "registros_por_ano": {str(k): int(v) for k, v in df["NU_ANO_REFERENCIA"].value_counts().items()},
    "resultados_modelos": resultados,
    "limitacao": "Separação temporal não impede sobreposição de curso/coorte entre treino e teste."
}
saida.write_text(json.dumps(relatorio, ensure_ascii=False, indent=2), encoding="utf-8")
print("Relatório de auditoria salvo em:", saida)


Relatório de auditoria salvo em: c:\Users\vivia\AM_SUP\relatorios\auditoria_notebook.json
